# DCC-GARCH Minimum-Variance Allocation

Uses adjusted closing prices for seven large-cap US stocks to estimate dynamic covariances and compare unconstrained minimum-variance weights with equal weights. The model is fitted on the full evaluation sample and uses same-period weights. Treat the results as an in-sample illustration of the allocation method.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from arch import arch_model
from scipy.optimize import minimize

# ------------------------------------------------------------
# Price history and returns
# ------------------------------------------------------------
MAG7 = ["AAPL", "MSFT", "AMZN", "GOOGL", "META", "NVDA", "TSLA"]

def load_prices(tickers=MAG7, start="2015-01-01", end=None):
    prices = yf.download(
        tickers, start=start, end=end, auto_adjust=True, group_by="column",
        multi_level_index=True, threads=False, progress=False, timeout=20,
    )
    if prices is None or prices.empty or "Close" not in prices.columns.get_level_values(0):
        raise RuntimeError("Yahoo Finance returned no adjusted close prices.")
    close = prices["Close"].reindex(columns=tickers).dropna().sort_index()
    if close.empty:
        raise RuntimeError("Yahoo Finance returned no dates with prices for all requested stocks.")
    return close

def to_log_returns(prices, freq=252):
    ret = np.log(prices).diff().dropna()
    return ret

# ------------------------------------------------------------
# Univariate GARCH(1,1) with Student-t innovations
# ------------------------------------------------------------
def fit_univariate_garch(returns_df):
    models = {}
    cond_vol = pd.DataFrame(index=returns_df.index, columns=returns_df.columns, dtype=float)
    z = pd.DataFrame(index=returns_df.index, columns=returns_df.columns, dtype=float)  # Standardized residuals

    for col in returns_df.columns:
        r = returns_df[col].astype(float) * 100.0  # Scale to percentage points for numerical stability
        am = arch_model(r, vol="GARCH", p=1, q=1, dist="t", mean="Constant")
        res = am.fit(disp="off")
        sigma_t = res.conditional_volatility.values / 100.0  # Convert back to return units
        cond_vol[col] = sigma_t
        z[col] = (returns_df[col].values) / np.where(sigma_t == 0, 1e-8, sigma_t)
        models[col] = res
    return models, cond_vol, z

# ------------------------------------------------------------
# Estimate DCC(1,1) by quasi-maximum likelihood
#    Q_t = (1-a-b) S + a * (e_{t-1} e_{t-1}^T) + b * Q_{t-1}
#    R_t = diag(Q_t)^{-1/2} Q_t diag(Q_t)^{-1/2}
# ------------------------------------------------------------
def dcc_loglik(params, z_mat):
    a, b = params
    # Penalize parameters outside the stationary region
    if a < 0 or b < 0 or (a + b) >= 0.999:
        return 1e6 + 1e6 * (abs(a) + abs(b) + max(0.0, a + b - 0.999))

    T, N = z_mat.shape
    S = np.cov(z_mat.T)
    Q = S.copy()
    inv_sqrt = lambda M: np.diag(1.0 / np.sqrt(np.clip(np.diag(M), 1e-12, None)))

    ll = 0.0
    for t in range(1, T):
        eps = z_mat[t - 1][:, None]
        Q = (1 - a - b) * S + a * (eps @ eps.T) + b * Q
        Dinv = inv_sqrt(Q)
        R = Dinv @ Q @ Dinv
        R = (R + R.T) / 2.0
        try:
            sign, logdet = np.linalg.slogdet(R)
            if sign <= 0:
                return 1e6
            invR = np.linalg.inv(R)
        except np.linalg.LinAlgError:
            return 1e6
        zt = z_mat[t][:, None]
        ll += logdet + (zt.T @ invR @ zt).item()
    return ll

def fit_dcc(z_df, a0=0.02, b0=0.97):
    z_mat = z_df.values.astype(float)
    x0 = np.array([a0, b0])
    bounds = [(1e-6, 0.999), (1e-6, 0.999)]
    res = minimize(dcc_loglik, x0, args=(z_mat,), method="L-BFGS-B", bounds=bounds)
    a, b = res.x
    return {"a": float(a), "b": float(b), "success": bool(res.success), "opt": res}

# ------------------------------------------------------------
# Reconstruct correlations and covariances
# ------------------------------------------------------------
def dcc_filter(z_df, cond_vol_df, a, b):
    idx = z_df.index
    cols = z_df.columns
    z_mat = z_df.values
    T, N = z_mat.shape

    S = np.cov(z_mat.T)
    Q = S.copy()

    def inv_sqrt_diag(M):
        return np.diag(1.0 / np.sqrt(np.clip(np.diag(M), 1e-12, None)))

    R_list = []
    Sigma_list = []

    for t in range(T):
        if t > 0:
            eps = z_mat[t - 1][:, None]
            Q = (1 - a - b) * S + a * (eps @ eps.T) + b * Q
        Dinv = inv_sqrt_diag(Q)
        R = Dinv @ Q @ Dinv
        R = (R + R.T) / 2.0

        Dt = np.diag(cond_vol_df.iloc[t].values)
        Sigma_t = Dt @ R @ Dt

        R_list.append(R)
        Sigma_list.append(Sigma_t)

    R_t = {idx[t]: pd.DataFrame(R_list[t], index=cols, columns=cols) for t in range(T)}
    Sigma_t = {idx[t]: pd.DataFrame(Sigma_list[t], index=cols, columns=cols) for t in range(T)}
    return R_t, Sigma_t

# ------------------------------------------------------------
# Compute daily minimum-variance weights
# ------------------------------------------------------------
def min_var_weights(Sigma_t_dict):
    dates = list(Sigma_t_dict.keys())
    cols = Sigma_t_dict[dates[0]].columns
    W = pd.DataFrame(index=dates, columns=cols, dtype=float)

    for dt in dates:
        S = Sigma_t_dict[dt].values
        N = S.shape[0]
        iota = np.ones((N, 1))
        try:
            Sinv = np.linalg.inv(S + 1e-10 * np.eye(N))
        except np.linalg.LinAlgError:
            Sinv = np.linalg.pinv(S + 1e-8 * np.eye(N))
        num = Sinv @ iota
        den = (iota.T @ Sinv @ iota).item()
        w = (num / den).ravel()
        W.loc[dt] = w
    return W

# ------------------------------------------------------------
# Compare in-sample results with equal weighting
# ------------------------------------------------------------
def backtest(weights_df, returns_df, freq=252, rf=0.0):
    w = weights_df.reindex(returns_df.index).ffill().dropna()
    r = returns_df.loc[w.index]
    strat_r = (w * r).sum(axis=1)
    mean = strat_r.mean() * freq
    vol = strat_r.std(ddof=1) * np.sqrt(freq)
    sharpe = (mean - rf) / (vol + 1e-12)
    eqw = pd.Series(1.0 / r.shape[1], index=r.columns)
    eqw_r = (eqw * r).sum(axis=1)
    eqw_mean = eqw_r.mean() * freq
    eqw_vol = eqw_r.std(ddof=1) * np.sqrt(freq)
    eqw_sharpe = (eqw_mean - rf) / (eqw_vol + 1e-12)
    out = {
        "annual_return": float(mean),
        "annual_vol": float(vol),
        "sharpe": float(sharpe),
        "eqw_annual_return": float(eqw_mean),
        "eqw_annual_vol": float(eqw_vol),
        "eqw_sharpe": float(eqw_sharpe),
        "curve": (1 + strat_r).cumprod(),
        "eqw_curve": (1 + eqw_r).cumprod(),
    }
    return out

# ------------------------------------------------------------
# Run the model and portfolio comparison
# ------------------------------------------------------------
def run_pipeline(start="2018-01-01", end=None, freq=252):
    prices = load_prices(MAG7, start=start, end=end)
    rets = to_log_returns(prices, freq=freq)
    _, cond_vol, z = fit_univariate_garch(rets)
    dcc_fit = fit_dcc(z, a0=0.02, b0=0.97)
    a, b = dcc_fit["a"], dcc_fit["b"]
    R_t, Sigma_t = dcc_filter(z, cond_vol, a, b)
    W = min_var_weights(Sigma_t)
    bt = backtest(W, rets, freq=freq)
    return {
        "prices": prices,
        "returns": rets,
        "cond_vol": cond_vol,
        "z": z,
        "dcc_params": {"a": a, "b": b, "success": dcc_fit["success"]},
        "R_t": R_t,
        "Sigma_t": Sigma_t,
        "weights": W,
        "backtest": bt,
    }

if __name__ == "__main__":
    results = run_pipeline(start="2019-01-01")
    print("DCC parameters:", results["dcc_params"])
    print("In-sample DCC Sharpe ratio:", results["backtest"]["sharpe"])
    print("In-sample equal-weight Sharpe ratio:", results["backtest"]["eqw_sharpe"])
